In [ ]:
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from rlm_sec.filings import sec_data
# from rlm_sec.filings.utils import company_to_ticker
# import asyncio

# from rlm_sec.trainer import hf_dataloader

# all_tickers_years = [(i,j) for i,j in zip(list(combined_qa['ticker_or_company_name']),list(combined_qa['year']))]
# all_tickers_years = list(set(all_tickers_years))
# combined_qa = hf_dataloader.load_combined_qa()

# # Function to call sec_main for a given (ticker, year)
# def fetch_sec_main(args):
#     ticker, year = args
#     ticker = company_to_ticker(ticker)
#     # sec_main is async, so run it with asyncio
#     if not ticker:
#         return None, None, None
#     return (ticker, year, asyncio.run(sec_data.sec_main(ticker, year)))

# results = []
# with ThreadPoolExecutor() as executor:
#     # Submit all (ticker, year) pairs for execution
#     futures = [executor.submit(fetch_sec_main, (ticker, year)) for ticker, year in all_tickers_years]
#     for future in as_completed(futures):
#         try:
#             ticker, year, value = future.result()
#             if not ticker:
#                 continue
#             results.append((ticker, year, value))
#         except Exception as e:
#             print(f"Error fetching ({ticker}, {year}): {e}")


In [ ]:
# import re
# from pathlib import Path
# from settings import env_settings
# from rlm_sec.dataloader.vector_store import FaissVectorIndex

# # Root containing one directory per "TICKER-YYYY" with *.md inside each.
# MARKDOWN_ROOT = Path("localworkspace/markdown/sec_data/")
# FORCE_REBUILD = False

# # Directory names must end with -YYYY (handles tickers like BRK-B-2025).
# _TICKER_YEAR_DIR = re.compile(r"^(?P<ticker>.+)-(?P<year>\d{4})$")

# index = FaissVectorIndex()
# all_keys = []

# for sub in sorted(MARKDOWN_ROOT.iterdir()):
#     if not sub.is_dir():
#         continue
#     m = _TICKER_YEAR_DIR.match(sub.name)
#     if not m:
#         print(f"skip (not TICKER-YYYY): {sub.name}")
#         continue
#     ticker, year = m["ticker"], m["year"]
#     md_paths = sorted(sub.glob("*.md"))
#     if not md_paths:
#         print(f"skip (no .md): {sub.name}")
#         continue
#     try:
#         keys = index.from_markdown(
#             ticker=ticker,
#             year=year,
#             markdown_paths=md_paths,
#             force=True,
#         )
#     except Exception:
#         pass
#     all_keys.extend(keys)
#     print(f"indexed {sub.name}: {len(keys)} filing(s)")

# print(f"total index keys: {len(all_keys)}")

## TESTING ENVIRONMENT

In [1]:
from rlm_sec.trainer import hf_dataloader

combined_qa = hf_dataloader.load_combined_qa()
# all_tickers_years = [(i,j) for i,j in zip(list(combined_qa['ticker_or_company_name']),list(combined_qa['year']))]
# all_tickers_years = list(set(all_tickers_years))

/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
combined_qa[0]

{'prompt': [{'role': 'system',
   'content': 'You are a helpful and harmless assistant.'},
  {'role': 'user',
   'content': 'Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you find you lack some knowledge, you can call a search engine using the format <search> query, ticker, year, filing_type </search>. Here, ticker refers to the company stock symbol (e.g., AAPL), year refers to the filing year, and filing_type specifies the SEC filing type. Filing types include: 10-K (annual report), 10-Q1 (quarter 1), 10-Q2 (quarter 2), 10-Q3 (quarter 3), and other variants if applicable. The search engine will return results between <information> and </information>. You can search as many times as needed. Once you have sufficient information, provide the final answer inside <answer> and </answer> without additional explanation. For example, <answer> Beijing </answer>. Question: What area did NVIDIA initi

In [ ]:
from datasets import load_dataset

# Load the "validation.parquet" file from the local directory using HuggingFace datasets
dataset = load_dataset("parquet", data_files="data/searchR1/validation.parquet")["train"]

dataset

Generating train split: 51713 examples [00:00, 72419.97 examples/s]

Dataset({
    features: ['data_source', 'prompt', 'ability', 'env_class', 'reward_spec', 'extra_info', 'metadata'],
    num_rows: 51713
})


In [16]:
dataset[0]

{'data_source': 'searchR1_nq',
 'prompt': [{'content': 'You are a helpful and harmless assistant.',
   'role': 'system'},
  {'content': 'Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search> and it will return the top searched results between <information> and </information>. You can search as many times as you want. If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer>, without detailed illustrations. For example, <answer> Beijing </answer>. Question: who got the first nobel prize in physics?',
   'role': 'user'}],
 'ability': 'fact-reasoning',
 'env_class': 'search',
 'reward_spec': {'ground_truth': {'target': ['Wilhelm Conrad Röntgen']},
  'style': 'rule'},
 'extra_info': {'index': 0,
  'need_tools_kwargs': True,
  'question': 'who got the firs